# The Reporting Layer

This notebook transforms our **Data Warehouse** into an **Analytics Application**.
It reads specific `reporting_*.sql` files, executes them in the database, validates the semantic models, and records performance.

In [ ]:
# --- 1. Setup & Connection ---
import os
import sys
import time
from pathlib import Path
import pandas as pd
from sqlalchemy import text

# Locate project root to import ETL config
PROJECT_ROOT = Path(".").resolve()
for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (p / "ETL").exists() or (p / "requirements.txt").exists():
        PROJECT_ROOT = p
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ETL.config import get_db_engine

try:
    engine = get_db_engine()
    print("Reporting execution engine connected successfully.")
except Exception as e:
    engine = None
    print(f"Database connection warning: {e}")

In [ ]:
# --- 2. Advanced Validation Framework ---

def validate_table(table_name: str, conn) -> dict:
    """Runs structural, business, and data quality checks."""
    results = {"status": "PASS", "messages": []}
    
    def log_error(msg):
        results["status"] = "FAIL"
        results["messages"].append(msg)
        
    try:
        # 1. Structural Check: Table Exists & Row Count
        count_res = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
        if count_res == 0:
            log_error("Data Quality: Table is empty (0 rows).")
            return results  # Stop further checks if empty
            
        # Get columns
        columns_res = conn.execute(text(f"SHOW COLUMNS FROM {table_name}")).fetchall()
        columns = [col[0].lower() for col in columns_res]
        
        # Null checks for grouping columns
        grouping_cols_map = {
            "reporting_sales_summary": "order_month",
            "reporting_category_summary": "product_category",
            "reporting_state_summary": "state_code",
            "reporting_customer_summary": "customer_unique_id"
        }
        if table_name in grouping_cols_map:
            grp_col = grouping_cols_map[table_name]
            if grp_col in columns:
                null_keys = conn.execute(text(f"SELECT COUNT(*) FROM {table_name} WHERE {grp_col} IS NULL")).scalar()
                if null_keys > 0:
                    log_error(f"Structural Check: Found {null_keys} NULL values in primary grouping column '{grp_col}'.")
                
                dups = conn.execute(text(f"SELECT COUNT(*) FROM (SELECT {grp_col} FROM {table_name} GROUP BY {grp_col} HAVING COUNT(*) > 1) t")).scalar()
                if dups > 0:
                    log_error(f"Structural Check: Found {dups} duplicate grouping keys for '{grp_col}'.")
        
        # Business Validation
        # Revenue
        if "total_revenue" in columns or "lifetime_revenue" in columns:
            rev_col = "total_revenue" if "total_revenue" in columns else "lifetime_revenue"
            dw_rev = conn.execute(text("SELECT SUM(total_sales_amount) FROM fact_sales")).scalar() or 0.0
            rpt_rev = conn.execute(text(f"SELECT SUM({rev_col}) FROM {table_name}")).scalar() or 0.0
            if abs(float(dw_rev) - float(rpt_rev)) > 0.1:
                log_error(f"Business Check: Revenue mismatch (DW: {dw_rev:.2f} vs RPT: {rpt_rev:.2f})")
            
            neg_rev = conn.execute(text(f"SELECT COUNT(*) FROM {table_name} WHERE {rev_col} < 0")).scalar()
            if neg_rev > 0:
                log_error(f"Data Quality: Found {neg_rev} rows with negative revenue.")
                
        # Orders
        if "total_orders" in columns:
            dw_orders = conn.execute(text("SELECT COUNT(DISTINCT order_id) FROM fact_sales")).scalar() or 0
            rpt_orders = conn.execute(text(f"SELECT SUM(total_orders) FROM {table_name}")).scalar() or 0
            if int(dw_orders) != int(rpt_orders):
                log_error(f"Business Check: Orders mismatch (DW: {dw_orders} vs RPT: {rpt_orders})")
                
        # Customers
        if "total_customers" in columns:
            dw_cust = conn.execute(text("SELECT COUNT(DISTINCT customer_key) FROM fact_sales")).scalar() or 0
            rpt_cust = conn.execute(text(f"SELECT SUM(total_customers) FROM {table_name}")).scalar() or 0
            if int(dw_cust) != int(rpt_cust):
                log_error(f"Business Check: Customers mismatch (DW: {dw_cust} vs RPT: {rpt_cust})")
                
    except Exception as e:
        log_error(f"Validation logic error: {e}")
        
    return results

In [ ]:
# --- 3. Execution Engine ---

REPORTING_SQL_FILES = [
    "reporting_sales_summary.sql",
    "reporting_category_summary.sql",
    "reporting_state_summary.sql",
    "reporting_customer_summary.sql",
]

sql_dir = PROJECT_ROOT / "SQL"
execution_metrics = []

if not engine:
    print("Cannot run engine: No database connection.")
else:
    print(f"Executing {len(REPORTING_SQL_FILES)} reporting SQL files...")
    
    for filename in REPORTING_SQL_FILES:
        file_path = sql_dir / filename
        table_name = filename.replace(".sql", "")
        
        if not file_path.exists():
            print(f"File not found: {filename}. Skipping.")
            execution_metrics.append({
                "Table": table_name,
                "Rows": 0,
                "Time (s)": 0.0,
                "Status": "❌",
                "Validation": "FAIL",
                "Notes": "File not found"
            })
            continue
            
        with open(file_path, 'r') as f:
            sql_script = f.read()
            
        # Split by semicolon
        statements = [s.strip() for s in sql_script.split(';') if s.strip()]
        
        start_time = time.perf_counter()
        status = "✅"
        validation_status = "PASS"
        error_msg = ""
        rows_processed = 0
        
        try:
            with engine.begin() as conn:
                for stmt in statements:
                    conn.execute(text(stmt))
                
                # Collect basic row count to confirm creation
                rows_processed = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
                
                # Run validation
                val_res = validate_table(table_name, conn)
                if val_res["status"] == "FAIL":
                    validation_status = "FAIL"
                    error_msg = " | ".join(val_res["messages"])
        except Exception as e:
            status = "❌"
            validation_status = "FAIL"
            error_msg = str(e)
            
        exec_time = time.perf_counter() - start_time
        
        execution_metrics.append({
            "Table": table_name,
            "Rows": rows_processed,
            "Time (s)": round(exec_time, 4),
            "Status": status,
            "Validation": validation_status,
            "Notes": error_msg
        })
        
        # Print inline status
        inline_status = "PASS" if status == "✅" and validation_status == "PASS" else "FAILED"
        print(f"{table_name:<35} {inline_status}")
        
    print("\nExecution complete.")

In [ ]:
# --- 4. Final Summary ---

df_summary = pd.DataFrame(execution_metrics)
# Reorder columns slightly for better presentation
cols = ["Table", "Rows", "Time (s)", "Status", "Validation"]
if "Notes" in df_summary.columns and (df_summary["Notes"] != "").any():
    cols.append("Notes")
df_summary = df_summary[cols]
df_summary